# W7D2 — Build the Vector Index — Lab

**Week 7 · Day 2 · Retrieval, RAG and Recommenders** · Lab

Yesterday you scored every chunk against every question with one matrix product. That works for
460 chunks and is impossible for two million, and this morning showed you the structure that
replaces it: cluster the vectors once, then search only the clusters near the query.

Today you build it three ways — brute force, an approximate FAISS index, and a persisted Chroma
store — and you measure the one number almost nobody measures: **how much recall the approximate
index gave up** to be fast.

The store you persist is not a demo. Wednesday's assistant opens it, and so does Friday's search.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٧ اليوم ٢ — ابنِ فهرس المتّجهات

**الأسبوع السابع · اليوم الثاني · الاسترجاع والتوليد المعزّز والتوصية** · معمل

بالأمس قِست كل مقطع مقابل كل سؤال بضرب مصفوفة واحد. وهذا يصلح لأربعمئة وستين مقطعًا ويستحيل
لمليونين، وقد أراك الصباح البنية التي تحلّ محلّه: عنقِد المتّجهات مرّة، ثم ابحث في العناقيد القريبة
من الاستعلام فقط.

واليوم تبنيه بثلاث طرق — بحثًا شاملًا، وفهرس FAISS تقريبيًّا، ومخزن Chroma محفوظًا — وتقيس الرقم الذي
لا يكاد أحد يقيسه: **كم استدعاءً تنازل عنه الفهرس التقريبي** ليكون سريعًا.

والمخزن الذي تحفظه ليس عرضًا للتجربة: يفتحه مساعد الأربعاء، ويفتحه بحث الجمعة.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Say what an approximate index actually does, and count the operations it saved.
- Measure **recall@5 of an index against brute force**, and refuse to ship one without that number.
- Read the `nprobe` dial as what it is: an accuracy/speed trade-off you choose, not a default.
- Persist a vector store with metadata and reopen it in a fresh process.
- Filter a query by metadata, and name a case where the filter removes the answer.
- Say when dense retrieval loses to keyword search, and show it on a query where it does.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقول ما الذي يفعله الفهرس التقريبي فعلًا، وأن تعدّ العمليات التي وفّرها.
- أن تقيس **استدعاء الفهرس عند ٥ مقابل البحث الشامل**، وألّا تسلّم فهرسًا بلا هذا الرقم.
- أن تقرأ مقبض `nprobe` على حقيقته: مقايضة بين الدقّة والسرعة تختارها أنت لا قيمة افتراضية.
- أن تحفظ مخزن متّجهات ببياناته الوصفية وأن تعيد فتحه في عملية جديدة.
- أن ترشّح استعلامًا بالبيانات الوصفية، وأن تسمّي حالة يحذف فيها المرشّح الإجابة نفسها.
- أن تقول متى يخسر الاسترجاع الكثيف أمام البحث بالكلمات، وأن تُريه على استعلام يخسره فعلًا.

</div>

## About the data

You are not loading a new dataset today. You load **yesterday's artefact**:
`chunk_comparison.parquet` and `chunks.parquet`, produced by W7D1 over the 24 `policy_docs`
documents. If you did not finish yesterday, `load_artefact` falls back to the reference copy in
`shared/solutions_cache/` and nothing here is blocked.

**The chunking strategy is not chosen for you.** The notebook reads yesterday's comparison table,
takes the strategy with the highest mean context recall, and indexes that one. Your index is built
on your own measurement.

**One thing added at indexing time.** Each chunk's *indexed* text is prefixed with its document id,
title and section — `POL-114 Instalment Plans and Missed Payments (POL-114 §2). This section …`.
That prefix is not decoration: it is what makes an identifier like `POL-114` findable at all, and
task 6 measures exactly that. Real systems do this and call it contextual chunk headers.

**The known problem:** 24 documents give roughly 85 chunks, and an approximate index over 85 vectors
is a toy. Its recall curve is real; its *timing* is not, because everything finishes in a tenth of a
millisecond. Task 3 therefore measures recall on the real corpus and speed on a 100,000-vector
synthetic set, and says which number came from where.

**First-run downloads:** `all-MiniLM-L6-v2` (~90 MB, cached from last week) and, in the stretch
section only, `paraphrase-MiniLM-L6-v2` (~90 MB).

<div dir="rtl" align="right">

## عن البيانات

لا تُحمّل بياناتٍ جديدة اليوم، بل تُحمّل **مُخرَج الأمس**: `chunk_comparison.parquet`
و`chunks.parquet` اللذين أنتجهما اليوم الأول على وثائق `policy_docs` الأربع والعشرين. وإن لم تُكمل
بالأمس فإن `load_artefact` يرجع إلى النسخة المرجعية في `shared/solutions_cache/` ولا شيء هنا يتوقّف.

**ولا تُختار طريقة التقطيع نيابةً عنك.** فالدفتر يقرأ جدول مقارنة الأمس، ويأخذ الطريقة الأعلى في
متوسّط استدعاء السياق، ويفهرسها. ففهرسك مبنيّ على قياسك أنت.

**وشيء واحد يُضاف وقت الفهرسة:** يُسبَق نصّ كل مقطع *المفهرَس* بمعرّف وثيقته وعنوانها وقسمه — مثل
`POL-114 Instalment Plans and Missed Payments (POL-114 §2). This section …`. وهذه السابقة ليست
زينة: بها وحدها يصير معرّف مثل `POL-114` قابلًا للإيجاد، والمهمة السادسة تقيس ذلك بالضبط. والأنظمة
الحقيقية تفعل هذا وتسمّيه ترويسات السياق للمقاطع.

**والمشكلة المعروفة:** أربع وعشرون وثيقة تعطي نحو خمسة وثمانين مقطعًا، والفهرس التقريبي على خمسة
وثمانين متّجهًا لعبة. منحنى استدعائه حقيقي، أما **زمنه** فلا، لأن كل شيء ينتهي في عُشر جزء من الألف
من الثانية. فتقيس المهمة الثالثة الاستدعاء على المُدوّنة الحقيقية والسرعة على مجموعة اصطناعية من مئة
ألف متّجه، وتقول أي رقم جاء من أين.

**تنزيل عند أول تشغيل:** `all-MiniLM-L6-v2` (نحو ٩٠ ميغابايت، مخزَّن من الأسبوع الماضي)، وفي قسم
التوسّع وحده `paraphrase-MiniLM-L6-v2` (نحو ٩٠ ميغابايت).

</div>

## Setup

**Two lines in the setup cell exist because of a real crash**, and they are worth reading.

`faiss` and `torch` each ship their own copy of the OpenMP runtime. On macOS, loading both into
one process and then running a threaded FAISS search kills the kernel — silently, with no
traceback, usually in the middle of a search you have run successfully ten times. `KMP_DUPLICATE_LIB_OK`
tells the runtime to tolerate the duplicate, and `faiss.omp_set_num_threads(1)` keeps FAISS on one
thread. The cost is that FAISS runs single-threaded, which for an index this size is nothing, and
the benefit is that the timings you record measure the algorithm rather than a thread pool.

If your kernel ever dies with no error during a FAISS call, this is why.

<div dir="rtl" align="right">

## الإعداد

**سطران في خلية الإعداد موجودان بسبب انهيار حقيقي**، ويستحقّان القراءة.

فمكتبتا `faiss` و`torch` تحمل كلٌّ منهما نسختها من زمن تشغيل OpenMP. وعلى macOS يؤدّي تحميلهما معًا
في عملية واحدة ثم تشغيل بحث FAISS متعدّد الخيوط إلى قتل النواة — بصمت وبلا أثر، وغالبًا في منتصف
بحثٍ نجح عشر مرات قبله. فـ`KMP_DUPLICATE_LIB_OK` تخبر زمن التشغيل أن يحتمل الازدواج، و
`faiss.omp_set_num_threads(1)` تُبقي FAISS على خيط واحد. والثمن أن يعمل FAISS بخيط واحد، وهو لا شيء
عند فهرس بهذا الحجم، والعائد أن الأزمنة التي تسجّلها تقيس الخوارزمية لا مجموعة الخيوط.

فإن ماتت نواتك يومًا بلا خطأ أثناء نداء FAISS فهذا هو السبب.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # faiss and torch both ship OpenMP

try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report
from aiep.viz import use_course_style, savefig

ensure("sentence-transformers", "faiss-cpu", "chromadb", "rank-bm25", "matplotlib",
       "pandas", "pyarrow")
seed_everything(42)

import shutil
import time

import chromadb
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

faiss.omp_set_num_threads(1)                            # see the markdown above — not optional
use_course_style()
np.set_printoptions(precision=3, suppress=True)

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
SECOND_EMBEDDER = "sentence-transformers/paraphrase-MiniLM-L6-v2"   # stretch section only
NLIST = 8                       # inverted lists — 8 over ~85 vectors is already coarse
TOP_K = 5
STORE_PATH = ARTEFACT_DIR / "chroma_store"
COLLECTION = "policy_chunks"

CORPUS_DIR = get_dataset_dir("policy_docs")
MANIFEST = pd.read_csv(CORPUS_DIR / "manifest.csv").set_index("doc_id")
QUESTIONS = pd.read_parquet(get_dataset("rag_eval_questions"))

# Yesterday's work — local artefacts/ first, then the reference copy.
COMPARISON = pd.read_parquet(load_artefact("chunk_comparison.parquet"))
ALL_CHUNKS = pd.read_parquet(load_artefact("chunks.parquet"))

MEAN_RECALL = (COMPARISON[COMPARISON.answerable]
               .groupby("strategy").recall_at_3.mean().sort_values(ascending=False))
WINNER = MEAN_RECALL.idxmax()
CHUNKS = ALL_CHUNKS[ALL_CHUNKS.strategy == WINNER].reset_index(drop=True)

print("yesterday's strategies, by mean context recall@3:")
print(MEAN_RECALL.round(3).to_string())
print(f"\nindexing '{WINNER}' — {len(CHUNKS)} chunks over {CHUNKS.doc_id.nunique()} documents")
print(versions(), "| device:", device())

## Section 1 — Warm-up: nine points, two queries  (≈25 min)

This morning's nine points, hard-coded, and the two searches that make the whole argument:

```
cluster 1        cluster 2        cluster 3
A (1,1)          D (4,1)          G (2,7)
B (2,1)          E (7,2)          H (3,8)
C (1,2)          F (8,1)          I (2,8)
```

**Query 1 — `q₁ = (7.6, 1.2)`.** Brute force checks all nine points and returns **F at 0.447**.
The clustered search checks three centroids and then the three points in the nearest list: **the
same answer in 6 operations instead of 9.** That is the whole promise of an index.

**Query 2 — `q₂ = (3.4, 1.0)`.** Brute force returns **D at 0.600**. The clustered search computes
the three centroid distances, finds `c₁` nearest at 2.09 — **and D is not in cluster 1.** It
returns **B at 1.400**. Correct code, wrong answer.

Then set `nprobe = 2`, search the two nearest lists, and get **D back — at 9 operations**, which is
brute force with extra steps. That is the dial: at one end it is fast and wrong, at the other it is
exact and pointless.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: تسع نقاط واستعلامان (نحو ٢٥ دقيقة)

نقاط الصباح التسع مكتوبةً في الرمز، والبحثان اللذان يقومان عليهما الحجّة كلها:

**الاستعلام الأول — `q₁ = (7.6, 1.2)`.** يفحص البحث الشامل النقاط التسع ويعيد **F عند 0.447**.
ويفحص البحث المعنقَد ثلاثة مراكز ثم النقاط الثلاث في أقرب قائمة: **الجواب نفسه في ٦ عمليات بدل ٩**.
وهذا وعد الفهرس كله.

**الاستعلام الثاني — `q₂ = (3.4, 1.0)`.** يعيد البحث الشامل **D عند 0.600**. ويحسب البحث المعنقَد
مسافات المراكز الثلاثة فيجد `c₁` أقربها عند 2.09 — **وD ليست في العنقود الأول**. فيعيد **B عند
1.400**. رمزٌ صحيح وجوابٌ خطأ.

ثم اضبط `nprobe = 2` وابحث في أقرب قائمتين فيعود **D — بتسع عمليات**، أي البحث الشامل وزيادة. وهذا
هو المقبض: عند طرفٍ سريعٌ مخطئ، وعند الآخر دقيقٌ بلا فائدة.

</div>

In [ ]:
POINTS = {"A": (1, 1), "B": (2, 1), "C": (1, 2),
          "D": (4, 1), "E": (7, 2), "F": (8, 1),
          "G": (2, 7), "H": (3, 8), "I": (2, 8)}
CLUSTERS = {"c1": "ABC", "c2": "DEF", "c3": "GHI"}
CENTROIDS = {name: np.mean([POINTS[p] for p in members], axis=0)
             for name, members in CLUSTERS.items()}
Q1, Q2 = np.array([7.6, 1.2]), np.array([3.4, 1.0])


def distances(query):
    return {name: float(np.linalg.norm(query - np.array(point, float)))
            for name, point in POINTS.items()}


def brute_force(query):
    """Check every point. Operations = one distance per point."""
    d = distances(query)
    winner = min(d, key=d.get)
    return winner, d[winner], len(POINTS)


def clustered(query, nprobe=1):
    """Three centroid distances, then only the points in the nprobe nearest lists."""
    to_centroid = {name: float(np.linalg.norm(query - c)) for name, c in CENTROIDS.items()}
    opened = sorted(to_centroid, key=to_centroid.get)[:nprobe]
    inside = {p: distances(query)[p] for name in opened for p in CLUSTERS[name]}
    winner = min(inside, key=inside.get)
    return winner, inside[winner], len(CENTROIDS) + len(inside), to_centroid


for label, query in [("q1", Q1), ("q2", Q2)]:
    point, dist, ops = brute_force(query)
    print(f"{label} brute force      → {point} at {dist:.3f}, {ops} operations")
    point, dist, ops, to_centroid = clustered(query, nprobe=1)
    print(f"{label} clustered nprobe=1 → {point} at {dist:.3f}, {ops} operations "
          f"(centroids {[round(v, 2) for v in to_centroid.values()]})")

WARM_Q2_WRONG = clustered(Q2, nprobe=1)[0] != brute_force(Q2)[0]
point, dist, ops, _ = clustered(Q2, nprobe=2)
print(f"\nq2 clustered nprobe=2 → {point} at {dist:.3f}, {ops} operations")
print(f"the index was wrong at nprobe=1: {WARM_Q2_WRONG} — and right at nprobe=2, "
      f"for the price of brute force")

## Section 2 — Core: six tasks  (≈60 min)

1. Embed the winning strategy's chunks, and check nothing was silently truncated.
2. Brute-force FAISS index — the ground truth everything else is measured against.
3. Approximate index, **recall@5 against brute force**, and the `nprobe` dial.
4. The same thing in Chroma, persisted, with metadata, reopened in a fresh client.
5. Metadata filtering — including a filter that removes the answer.
6. Dense against BM25 on five queries, three semantic and two identifiers.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. ضمّن مقاطع الطريقة الفائزة، وتحقّق أن شيئًا لم يُبتر بصمت.
٢. فهرس FAISS الشامل — المرجع الذي يُقاس عليه كل ما بعده.
٣. الفهرس التقريبي، و**استدعاؤه عند ٥ مقابل الشامل**، ومقبض `nprobe`.
٤. الشيء نفسه في Chroma، محفوظًا ببياناته الوصفية، ثم مُعادَ فتحه بعميل جديد.
٥. الترشيح بالبيانات الوصفية — ومنه مرشّح يحذف الإجابة.
٦. الكثيف مقابل BM25 على خمسة استعلامات، ثلاثة دلالية واثنان بمعرّفات.

</div>

### Task 2.1 — embed, and check for silent truncation

Build the indexed text for every chunk — `doc_id`, title, section, then the chunk — and embed it.
Record three things: the embedding dimension, the wall-clock, and the chunk count.

Then the part people skip. A sentence-transformer has a maximum sequence length, and text past it
is **dropped without a warning**. Tokenise every chunk, count how many exceed
`model.max_seq_length`, and report the fraction. If it is not zero, your index silently does not
contain the ends of those chunks — and the failure looks exactly like bad retrieval.

<div dir="rtl" align="right">

### المهمة ٢٫١ — ضمّن، وتحقّق من البتر الصامت

ابنِ النصّ المفهرَس لكل مقطع — المعرّف والعنوان والقسم ثم المقطع — وضمّنه. وسجّل ثلاثة أشياء: بُعد
التضمين، وزمن التنفيذ، وعدد المقاطع.

ثم الجزء الذي يتخطّاه الناس. فلنموذج الجمل طول تسلسل أقصى، وما بعده **يُحذف بلا تحذير**. فقطّع كل
مقطع إلى رموز، وعُدّ كم منها يتجاوز `model.max_seq_length`، واذكر النسبة. فإن لم تكن صفرًا فإن
فهرسك لا يحوي أواخر تلك المقاطع، والإخفاق يبدو تمامًا كإخفاق استرجاع.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build the indexed text as f"{doc_id} {title} ({sections}). {text}" — the title comes
#    from MANIFEST, which is indexed by doc_id.
# 2) model.encode(list_of_texts, normalize_embeddings=True) returns a float32 array;
#    time it with time.perf_counter() around the call, not around the whole cell.
# 3) model.tokenizer.encode(text) gives the token ids for one chunk. Compare len() of
#    that against model.max_seq_length and report the fraction over the limit.
# Search: "sentence transformers max_seq_length truncation"
# https://www.sbert.net/docs/package_reference/SentenceTransformer.html
#
# ١) ابنِ النصّ المفهرَس بالصيغة `f"{doc_id} {title} ({sections}). {text}"` — والعنوان من
#    `MANIFEST` المفهرس بالمعرّف.
# ٢) تُعيد `model.encode(texts, normalize_embeddings=True)` مصفوفة float32؛ وقِس زمنها
#    بـ`time.perf_counter()` حول النداء لا حول الخلية كلها.
# ٣) تعطيك `model.tokenizer.encode(text)` رموز مقطع واحد. قارن طولها بـ
#    `model.max_seq_length` واذكر نسبة ما تجاوز الحدّ.
# ابحث عن: "sentence transformers max_seq_length truncation"
# https://www.sbert.net/docs/package_reference/SentenceTransformer.html
# ────────────────────────────────────────────────────────────────────

model = SentenceTransformer(EMBEDDER)
# TODO: Build CHUNKS["indexed"]: the document id, title and section ids in front of the chunk.
# مهمة: ابنِ العمود `CHUNKS["indexed"]`: معرّف الوثيقة وعنوانها ومعرّفات القسم قبل نصّ المقطع.
# TODO: Embed every indexed chunk, normalised, and record how long it took.
# مهمة: ضمّن كل مقطع مفهرَس مع التطبيع، وسجّل كم استغرق.
# TODO: Count the chunks whose token length exceeds the model's maximum sequence length.
# مهمة: عُدّ المقاطع التي يتجاوز طولها بالرموز أقصى طول تسلسل للنموذج.
print(f"{len(CHUNKS)} chunks → {VECTORS.shape} in {EMBED_SECONDS:.2f}s "
      f"({len(CHUNKS) / EMBED_SECONDS:.0f} chunks/second)")
print(f"tokens per chunk: median {np.median(TOKEN_LENGTHS):.0f}, "
      f"max {TOKEN_LENGTHS.max()}, limit {model.max_seq_length}")
print(f"silently truncated: {TRUNCATED} chunks "
      f"({TRUNCATED / len(CHUNKS):.1%}) — anything above zero is text your index does not have")

### Task 2.2 — brute force first, because it is the ground truth

Build a FAISS `IndexFlatL2` over the chunk vectors and search all 40 questions for their top 5.
Flat is not a fallback; it is the **answer key**. Every recall number in task 3 is measured against
these results, and an approximate index with nothing to compare against is a system whose accuracy
nobody has ever established.

Record the query time as well, so task 3 has a baseline for the speed half of the trade-off.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الشامل أولًا، لأنه المرجع

ابنِ فهرس `IndexFlatL2` في FAISS على متّجهات المقاطع، وابحث عن أفضل خمسة لكل سؤال من الأربعين.
والفهرس المسطّح ليس خيارًا احتياطيًّا بل **مفتاح الإجابات**. فكل رقم استدعاء في المهمة الثالثة يُقاس
عليه، والفهرس التقريبي بلا مرجع نظامٌ لم يُثبت أحد دقّته قط.

وسجّل زمن الاستعلام أيضًا، ليكون للمهمة الثالثة أساسٌ في نصف السرعة من المقايضة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) faiss.IndexFlatL2(dimension) then .add(vectors). The dimension has to be a Python
#    int — VECTORS.shape[1] straight from numpy sometimes is not.
# 2) Encode all 40 questions once, normalised, exactly like the chunks. The same model
#    and the same normalisation, or the distances mean nothing.
# 3) index.search(queries, TOP_K) returns (distances, indices). Time only the search.
# Search: "faiss IndexFlatL2 search python"
# https://faiss.ai/cpp_api/struct/structfaiss_1_1IndexFlat.html
#
# ١) `faiss.IndexFlatL2(dimension)` ثم `.add(vectors)`. ويجب أن يكون البُعد عددًا صحيحًا
#    في بايثون — و`VECTORS.shape[1]` من numpy قد لا يكون كذلك.
# ٢) ضمّن الأسئلة الأربعين مرّة واحدة مع التطبيع، تمامًا كالمقاطع. النموذج نفسه والتطبيع
#    نفسه، وإلا فلا معنى للمسافات.
# ٣) تُعيد `index.search(queries, TOP_K)` المسافات والفهارس. وقِس زمن البحث وحده.
# ابحث عن: "faiss IndexFlatL2 search python"
# https://faiss.ai/cpp_api/struct/structfaiss_1_1IndexFlat.html
# ────────────────────────────────────────────────────────────────────

DIMENSION = int(VECTORS.shape[1])
QUERY_VECTORS = model.encode(QUESTIONS.question.tolist(),
                             normalize_embeddings=True, batch_size=64)
# TODO: Build the flat index, add the chunk vectors, and search the 40 questions for TOP_K.
# مهمة: ابنِ الفهرس المسطّح، وأضف متّجهات المقاطع، وابحث عن أفضل `TOP_K` للأسئلة الأربعين.
print(f"flat index: {FLAT.ntotal} vectors, {DIMENSION} dimensions, "
      f"{FLAT_MS:.3f} ms per query")
example = QUESTIONS.iloc[0]
print(f"\n{example.qid}: {example.question}")
for rank, chunk_id in enumerate(FLAT_IDS[0], start=1):
    print(f"  {rank}. {CHUNKS.doc_id[chunk_id]} {CHUNKS.sections[chunk_id]}")
print(f"  gold: {example.gold_sections}")

### Task 2.3 — the approximate index, and the number nobody reports

Build an `IndexIVFFlat` with `NLIST` inverted lists over the same vectors, then measure
**recall@5 against the flat index** at `nprobe` = 1, 2, 4 and `NLIST`. Recall@5 here is: of the
five chunks the flat index returned, how many did the approximate index also return, averaged over
the 40 questions.

You will see recall rise monotonically with `nprobe` and reach 1.0 when you probe every list —
because probing every list *is* brute force.

**Then the honest part.** Query time on 85 vectors is a tenth of a millisecond for every setting,
so this corpus cannot show you the speed half of the trade-off. Build a second index over 100,000
random 384-dimensional vectors, measure recall and time there, and plot that. Say clearly in the
output which numbers came from the real corpus and which from the synthetic one — reporting a
speed-up measured on random vectors as though it were your system's is a very common way to be
wrong in public.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الفهرس التقريبي، والرقم الذي لا يذكره أحد

ابنِ `IndexIVFFlat` بعدد `NLIST` من القوائم المقلوبة على المتّجهات نفسها، ثم قِس **الاستدعاء عند ٥
مقابل الفهرس المسطّح** عند `nprobe` = ١ و٢ و٤ و`NLIST`. والاستدعاء عند ٥ هنا هو: من المقاطع الخمسة
التي أعادها المسطّح، كم أعاد التقريبي منها، بمتوسّط الأسئلة الأربعين.

وسترى الاستدعاء يرتفع اطّرادًا مع `nprobe` ويبلغ ١٫٠ حين تفحص كل القوائم — لأن فحص كل القوائم **هو**
البحث الشامل.

**ثم الجزء الأمين.** زمن الاستعلام على خمسة وثمانين متّجهًا عُشر جزء من الألف من الثانية عند كل ضبط،
فهذه المُدوّنة لا تستطيع أن تريك نصف السرعة من المقايضة. فابنِ فهرسًا ثانيًا على مئة ألف متّجه عشوائي
بـ٣٨٤ بُعدًا، وقِس الاستدعاء والزمن هناك، وارسمهما. وقل في المخرجات صراحةً أي الأرقام من المُدوّنة
الحقيقية وأيّها من الاصطناعية — فذكر تسريعٍ مقيسٍ على متّجهات عشوائية كأنه تسريع نظامك طريقة شائعة
جدًّا للخطأ أمام الناس.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) quantiser = faiss.IndexFlatL2(dim); index = faiss.IndexIVFFlat(quantiser, dim,
#    NLIST). An IVF index must be .train(vectors) before .add(vectors).
# 2) Recall@5 for one question is len(set(approx_ids) & set(flat_ids)) / TOP_K. Average
#    it over the 40 questions, once per nprobe setting.
# 3) For the synthetic run, np.random.default_rng(42).random((100_000, 384)) cast to
#    float32 is enough. Build a flat index over it for ground truth, and time both.
# Search: "faiss IndexIVFFlat nprobe train recall"
# https://github.com/facebookresearch/faiss/wiki/Faster-search
#
# ١) `quantiser = faiss.IndexFlatL2(dim)` ثم `index = faiss.IndexIVFFlat(quantiser, dim,
#    NLIST)`. ويجب تدريب فهرس IVF بـ`.train(vectors)` قبل `.add(vectors)`.
# ٢) الاستدعاء عند ٥ لسؤال واحد هو `len(set(approx) & set(flat)) / TOP_K`. ومتوسّطه على
#    الأسئلة الأربعين، مرّة لكل قيمة `nprobe`.
# ٣) وللتجربة الاصطناعية يكفي `np.random.default_rng(42).random((100_000, 384))` محوَّلًا
#    إلى float32. وابنِ عليه فهرسًا مسطّحًا للمرجع، وقِس زمن الاثنين.
# ابحث عن: "faiss IndexIVFFlat nprobe train recall"
# https://github.com/facebookresearch/faiss/wiki/Faster-search
# ────────────────────────────────────────────────────────────────────

PROBES = [1, 2, 4, NLIST]
def recall_at_k(approximate_ids, exact_ids):
    """Of the exact top-k, how many did the approximate index also return?"""
    return float(np.mean([len(set(a) & set(b)) / len(b)
                          for a, b in zip(approximate_ids, exact_ids)]))
# TODO: Build the IVF index over the chunk vectors, then measure recall@5 and query time at every nprobe in PROBES.
# مهمة: ابنِ فهرس IVF على متّجهات المقاطع، ثم قِس الاستدعاء عند ٥ وزمن الاستعلام عند كل قيمة `nprobe` في `PROBES`.
print("the real corpus — recall is meaningful here, the timings are not:")
print(REAL_BENCH[["nprobe", "recall_at_5", "query_ms"]].round(3).to_string(index=False))

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Make 100,000 random float32 vectors of the same dimension and 200 random queries.
# 2) Build a flat index over them for the ground truth, and time one search — that is the
#    number the approximate index has to beat.
# 3) Reuse the same bench function with a larger nlist (316 ≈ sqrt(100,000) is the usual
#    rule of thumb) and plot recall against query time.
# Search: "faiss how many inverted lists nlist sqrt rule"
# https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index
#
# ١) ولّد مئة ألف متّجه عشوائي float32 بالبُعد نفسه، ومئتَي استعلام عشوائي.
# ٢) ابنِ عليها فهرسًا مسطّحًا للمرجع، وقِس زمن بحث واحد — وهو الرقم الذي على الفهرس
#    التقريبي أن يتفوّق عليه.
# ٣) أعِد استعمال دالة القياس نفسها بعدد قوائم أكبر (٣١٦ ≈ الجذر التربيعي لمئة ألف هي
#    القاعدة المعتادة) وارسم الاستدعاء مقابل زمن الاستعلام.
# ابحث عن: "faiss how many inverted lists nlist sqrt rule"
# https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index
# ────────────────────────────────────────────────────────────────────

# TODO: Repeat the benchmark on 100,000 synthetic vectors, where the query time is measurable.
# مهمة: أعِد القياس على مئة ألف متّجه اصطناعي، حيث يصير زمن الاستعلام قابلًا للقياس.
print(f"synthetic 100,000 vectors — brute force is {BIG_FLAT_MS:.2f} ms per query\n")
print(BIG_BENCH[["nprobe", "recall_at_5", "query_ms"]].round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(REAL_BENCH.nprobe, REAL_BENCH.recall_at_5, marker="o")
axes[0].set_xlabel("nprobe"); axes[0].set_ylabel("recall@5 vs brute force")
axes[0].set_title(f"policy_docs ({len(CHUNKS)} vectors)")
axes[1].plot(BIG_BENCH.query_ms, BIG_BENCH.recall_at_5, marker="o")
axes[1].axvline(BIG_FLAT_MS, linestyle="--", linewidth=1)
axes[1].set_xlabel("query time (ms)"); axes[1].set_ylabel("recall@5")
axes[1].set_title("synthetic 100k — dashed line is brute force")
savefig(fig, "index_tradeoff.png")
plt.show()

### Task 2.4 — Chroma, persisted, with the metadata attached

FAISS stores vectors and nothing else: search it and you get row numbers, and you keep the mapping
from row number to document yourself. Chroma stores the text and the metadata beside the vector,
persists to disk, and returns documents.

Write the same chunks into a `PersistentClient` at `STORE_PATH` with `doc_id`, `sections` and
`chunk_index` as metadata. Query it with five questions and check the results against the FAISS
flat index. They should agree on the top hit for all five — same vectors, same metric, so a
disagreement means one of the two is configured differently from what you think.

Then **close the client, open a new one on the same path, and query again**. That reopen is the
whole point of a persisted store, and it is what Wednesday's notebook does with no code from today.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — Chroma محفوظًا ببياناته الوصفية

يخزّن FAISS المتّجهات ولا شيء غيرها: تبحث فيه فتحصل على أرقام صفوف، وتتولّى أنت الربط بين رقم الصف
والوثيقة. أما Chroma فيخزّن النصّ والبيانات الوصفية بجوار المتّجه، ويحفظ إلى القرص، ويعيد وثائق.

اكتب المقاطع نفسها في `PersistentClient` عند `STORE_PATH` مع `doc_id` و`sections` و`chunk_index`
بيانات وصفية. واستعلمه بخمسة أسئلة وطابق النتائج مع فهرس FAISS المسطّح. والمفترض أن يتّفقا على أول
نتيجة في الخمسة جميعًا — فالمتّجهات نفسها والمقياس نفسه، والاختلاف يعني أن أحدهما مضبوط على غير ما
تظنّ.

ثم **أغلق العميل، وافتح عميلًا جديدًا على المسار نفسه، واستعلم ثانيةً**. وإعادة الفتح هذه هي مقصود
المخزن المحفوظ كله، وهي ما يفعله دفتر الأربعاء بلا سطر من رمز اليوم.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) chromadb.PersistentClient(path=str(STORE_PATH)) then
#    client.get_or_create_collection(COLLECTION, metadata={"hnsw:space": "cosine"}).
# 2) collection.add(ids=..., embeddings=..., documents=..., metadatas=...) — ids must be
#    unique strings, so "POL-114#3" (doc id and chunk index) is a natural choice.
# 3) collection.query(query_embeddings=[...], n_results=TOP_K) returns a dict of lists,
#    one list per query, with "ids", "documents", "metadatas" and "distances".
# Search: "chromadb PersistentClient add query metadata"
# https://docs.trychroma.com/docs/collections/manage-collections
#
# ١) `chromadb.PersistentClient(path=str(STORE_PATH))` ثم
#    `client.get_or_create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})`.
# ٢) `collection.add(ids=..., embeddings=..., documents=..., metadatas=...)` — والمعرّفات
#    نصوص فريدة، فـ`"POL-114#3"` (معرّف الوثيقة ورقم المقطع) اختيار طبيعي.
# ٣) تُعيد `collection.query(query_embeddings=[...], n_results=TOP_K)` قاموسًا من القوائم،
#    قائمة لكل استعلام، فيه `ids` و`documents` و`metadatas` و`distances`.
# ابحث عن: "chromadb PersistentClient add query metadata"
# https://docs.trychroma.com/docs/collections/manage-collections
# ────────────────────────────────────────────────────────────────────

CHUNK_IDS = [f"{row.doc_id}#{row.chunk_index}" for row in CHUNKS.itertuples()]
shutil.rmtree(STORE_PATH, ignore_errors=True)   # rebuild from scratch every run
# TODO: Create the persistent collection and add every chunk with its vector, text and metadata.
# مهمة: أنشئ المجموعة المحفوظة وأضف كل مقطع بمتّجهه ونصّه وبياناته الوصفية.
CHECK_QUESTIONS = QUESTIONS.head(5)
check_vectors = QUERY_VECTORS[:5]
# TODO: Query Chroma with those five and compare the top hit against the flat FAISS index.
# مهمة: استعلم Chroma بتلك الخمسة وقارن أول نتيجة بفهرس FAISS المسطّح.
for i, row in CHECK_QUESTIONS.reset_index(drop=True).iterrows():
    print(f"{row.qid} faiss {CHUNK_IDS[FLAT_IDS[i][0]]:<12} "
          f"chroma {chroma_result['ids'][i][0]:<12} "
          f"{'agree' if chroma_result['ids'][i][0] == CHUNK_IDS[FLAT_IDS[i][0]] else 'DISAGREE'}")
print(f"\ntop-1 agreement: {AGREEMENT}/{len(CHECK_QUESTIONS)}")
# The reopen. A fresh client, the same path, no re-embedding of anything.
del collection, client
reopened = chromadb.PersistentClient(path=str(STORE_PATH)).get_collection(COLLECTION)
REOPENED_COUNT = reopened.count()
REOPENED_MATCH = (reopened.query(query_embeddings=[check_vectors[0].tolist()],
                                 n_results=TOP_K)["ids"][0]
                  == chroma_result["ids"][0])
print(f"reopened store: {REOPENED_COUNT} chunks, same results as before: {REOPENED_MATCH}")

### Task 2.5 — metadata filtering, and the filter that deletes the answer

Run one query three ways: unfiltered, restricted to a single `doc_id`, and restricted to a
`doc_id` that does **not** contain the answer.

The middle one is the good case — a user who says "this is about my instalment plan" has given you
information, and using it makes retrieval better and cheaper. The third is the failure that gets
shipped: the filter is applied confidently, the store returns its best matches *within the filter*,
and they look like answers. Nothing errors. Nothing is empty. The answer is simply not in the
result, and no metric on the generation side can see why.

Print all three result sets and write one sentence on how a real system should decide whether to
filter at all.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الترشيح بالبيانات الوصفية، والمرشّح الذي يحذف الإجابة

شغّل استعلامًا واحدًا بثلاث صور: بلا ترشيح، ومقصورًا على `doc_id` واحد، ومقصورًا على `doc_id` **لا**
يحوي الإجابة.

والحالة الوسطى هي الحسنة — فالمستخدم الذي يقول «سؤالي عن خطة التقسيط» قد أعطاك معلومة، واستعمالها
يجعل الاسترجاع أفضل وأرخص. والثالثة هي الإخفاق الذي يُسلَّم للناس: يُطبَّق المرشّح بثقة، فيعيد المخزن
أفضل ما عنده **داخل المرشّح**، وتبدو النتائج إجابات. ولا خطأ يظهر، ولا نتيجة فارغة. الإجابة ببساطة
ليست في النتيجة، ولا يستطيع أي مقياس في جانب التوليد أن يرى السبب.

اطبع المجموعات الثلاث، واكتب جملة واحدة عن كيف يقرّر نظام حقيقي هل يرشّح أصلًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) collection.query(..., where={"doc_id": "POL-114"}) restricts the search to chunks
#    whose metadata matches. The filter is applied before the vector search, not after.
# 2) Pick a question whose gold section you know — QUESTIONS has gold_sections — then
#    filter to a document that is not in it.
# 3) Print the returned ids for all three runs side by side, and check whether the gold
#    document appears at all.
# Search: "chroma where metadata filter query"
# https://docs.trychroma.com/docs/querying-collections/metadata-filtering
#
# ١) `collection.query(..., where={"doc_id": "POL-114"})` يقصر البحث على المقاطع التي
#    تطابق بياناتها الوصفية. ويُطبَّق المرشّح قبل بحث المتّجهات لا بعده.
# ٢) اختر سؤالًا تعرف قسمه المرجعي — و`QUESTIONS` فيه `gold_sections` — ثم رشّح إلى وثيقة
#    ليست فيه.
# ٣) اطبع المعرّفات المُعادة في الحالات الثلاث جنبًا إلى جنب، وتحقّق هل تظهر الوثيقة
#    المرجعية أصلًا.
# ابحث عن: "chroma where metadata filter query"
# https://docs.trychroma.com/docs/querying-collections/metadata-filtering
# ────────────────────────────────────────────────────────────────────

FILTER_QUESTION = QUESTIONS[QUESTIONS.qid == "Q26"].iloc[0]    # POL-114 §2
filter_vector = model.encode([FILTER_QUESTION.question], normalize_embeddings=True)
gold_doc = FILTER_QUESTION.gold_docs
# TODO: Query the reopened collection three ways: no filter, filtered to the gold document, and filtered to a document that does not contain the answer.
# مهمة: استعلم المجموعة المُعاد فتحها بثلاث صور: بلا مرشّح، ومرشّحًا إلى الوثيقة المرجعية، ومرشّحًا إلى وثيقة لا تحوي الإجابة.
print(f"{FILTER_QUESTION.qid}: {FILTER_QUESTION.question}")
print(f"gold: {FILTER_QUESTION.gold_sections}\n")
for label, ids in FILTER_RUNS.items():
    found = any(chunk_id.startswith(gold_doc) for chunk_id in ids)
    print(f"  {label:<26} {ids}  gold document present: {found}")
print("\nThe third run returned three confident results and none of them can answer the "
      "question. Nothing raised, nothing was empty.")

### Task 2.6 — dense against keyword, on five queries

Five queries: three semantic and two that hang on a literal identifier.

| Query | Kind |
|---|---|
| `what happens if I miss a payment` | semantic |
| `how long do I have to return an item` | semantic |
| `can I work from home` | semantic |
| `policy POL-114` | identifier |
| `form 27B` | identifier |

Run each through the dense index and through BM25 over the same indexed text, and put the top
result for both in one table with the correct document beside it.

The dense retriever wins the sentences and **loses `policy POL-114`**: an embedding of a document
id is a point in a space where nothing is near it, because there is no meaning in `POL-114` to be
near. BM25 finds it immediately, because a rare token is exactly what BM25 rewards.

That table is the argument for hybrid retrieval, which you build on Thursday.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الكثيف مقابل الكلمات، على خمسة استعلامات

خمسة استعلامات: ثلاثة دلالية واثنان يتعلّقان بمعرّف حرفي.

شغّل كلًّا منها في الفهرس الكثيف وفي BM25 على النصّ المفهرَس نفسه، وضع أول نتيجة للاثنين في جدول
واحد بجواره الوثيقة الصحيحة.

يفوز المُسترجِع الكثيف في الجمل و**يخسر `policy POL-114`**: فتضمين معرّف وثيقة نقطةٌ في فضاء لا شيء
قريب منها، إذ لا معنى في `POL-114` ليقرب منه شيء. أما BM25 فيجده فورًا، لأن الرمز النادر هو بالضبط ما
يكافئه BM25.

وذلك الجدول هو الحجّة للاسترجاع الهجين الذي تبنيه يوم الخميس.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) BM25Okapi takes a list of token lists. Lower-case and strip punctuation the same
#    way for the corpus and for the query, or "POL-114." and "POL-114" are two tokens.
# 2) bm25.get_scores(query_tokens) returns one score per document; argsort the negated
#    scores for the ranking.
# 3) Score the dense side from the flat index you already built — encode the five
#    queries and search — and report the document id of the top hit for both.
# Search: "rank_bm25 BM25Okapi get_scores"
# https://github.com/dorianbrown/rank_bm25
#
# ١) تأخذ `BM25Okapi` قائمة من قوائم الرموز. وحوّل الحروف إلى صغيرة وانزع الترقيم بالطريقة
#    نفسها للمُدوّنة وللاستعلام، وإلا صار `"POL-114."` و`"POL-114"` رمزين.
# ٢) تُعيد `bm25.get_scores(query_tokens)` درجةً لكل وثيقة؛ ورتّب الدرجات المنفية بـ
#    `argsort` للحصول على الترتيب.
# ٣) واحسب الجانب الكثيف من الفهرس المسطّح الذي بنيته — ضمّن الاستعلامات الخمسة وابحث —
#    واذكر معرّف وثيقة أول نتيجة عند الاثنين.
# ابحث عن: "rank_bm25 BM25Okapi get_scores"
# https://github.com/dorianbrown/rank_bm25
# ────────────────────────────────────────────────────────────────────

PROBE_QUERIES = [("what happens if I miss a payment", "POL-114", "semantic"),
                 ("how long do I have to return an item", "POL-101", "semantic"),
                 ("can I work from home", "POL-110", "semantic"),
                 ("policy POL-114", "POL-114", "identifier"),
                 ("form 27B", "POL-111", "identifier")]
def tokenise(text):
    """Lower-case, strip the punctuation that would split an identifier off its token."""
    return [word.lower().strip(".,()§#") for word in text.split()]
# TODO: Build a BM25 index over the same indexed text, then score all five queries with both retrievers and record which document each one ranked first.
# مهمة: ابنِ فهرس BM25 على النصّ المفهرَس نفسه، ثم احسب درجات الاستعلامات الخمسة بالطريقتين وسجّل أي وثيقة رتّبتها كلٌّ منهما أولًا.
print(RETRIEVER_TABLE.to_string(index=False))
identifier = RETRIEVER_TABLE[RETRIEVER_TABLE.kind == "identifier"]
semantic = RETRIEVER_TABLE[RETRIEVER_TABLE.kind == "semantic"]
print(f"\nidentifier queries — dense {identifier.dense_correct.sum()}/{len(identifier)}, "
      f"BM25 {identifier.bm25_correct.sum()}/{len(identifier)}")
print(f"semantic queries   — dense {semantic.dense_correct.sum()}/{len(semantic)}, "
      f"BM25 {semantic.bm25_correct.sum()}/{len(semantic)}")

## Section 3 — Stretch: the cost of changing your mind  (≈30 min)

Two parts, and the second is a number to write down.

**(a) The wrong model.** Embed the five probe queries with a *different* model of the same
dimension and search the index you built with the first one. Nothing raises: the shapes match, the
distances are computed, results come back. They are just wrong. An index is not a set of vectors —
it is a set of vectors **plus the model that made them**, and losing track of that pairing is a
class of outage that is very hard to diagnose from the symptoms.

**(b) The re-embedding bill.** You measured the embedding time for your corpus in task 1.
Extrapolate: how long to re-embed 2 million documents on this machine? Print the hours. That number
is the answer to this morning's activity 2, and it is why "we will just switch to a better embedding
model" is a migration and not a config change.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: ثمن تغيير رأيك (نحو ٣٠ دقيقة)

جزآن، وثانيهما رقم تكتبه عندك.

**(أ) النموذج الخطأ.** ضمّن الاستعلامات الخمسة بنموذج *مختلف* من البُعد نفسه، وابحث في الفهرس الذي
بنيته بالأول. لا خطأ يُرفع: الأشكال متوافقة، والمسافات تُحسب، والنتائج تعود. لكنها خاطئة. فالفهرس
ليس مجموعة متّجهات بل مجموعة متّجهات **مع النموذج الذي صنعها**، وضياع هذا الاقتران صنفٌ من الأعطال
يصعب جدًّا تشخيصه من أعراضه.

**(ب) فاتورة إعادة التضمين.** قِست زمن التضمين لمُدوّنتك في المهمة الأولى. فاستقرِ منه: كم يلزم
لإعادة تضمين مليونَي وثيقة على هذا الجهاز؟ اطبع الساعات. وهذا الرقم جواب نشاط الصباح الثاني، وبه
تعرف لماذا «سننتقل إلى نموذج تضمين أفضل» هجرةٌ لا تغيير إعداد.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Load SECOND_EMBEDDER, encode the same five queries, and search FLAT with those
#    vectors. Compare the returned document ids against the right model's results.
# 2) Count how many of the five still return the correct document. That count is the
#    measurement — "it looks wrong" is not.
# 3) For the bill: chunks per second from task 1, then 2,000,000 / rate, in hours.
# Search: "embedding model migration reindex cost"
# https://www.sbert.net/docs/pretrained_models.html
#
# ١) حمّل `SECOND_EMBEDDER`، وضمّن الاستعلامات الخمسة نفسها، وابحث في `FLAT` بتلك
#    المتّجهات. وقارن معرّفات الوثائق المُعادة بنتائج النموذج الصحيح.
# ٢) وعُدّ كم من الخمسة ما زال يعيد الوثيقة الصحيحة. وهذا العدد هو القياس — و«تبدو خاطئة»
#    ليست قياسًا.
# ٣) وللفاتورة: عدد المقاطع في الثانية من المهمة الأولى، ثم ٢٬٠٠٠٬٠٠٠ ÷ المعدّل، بالساعات.
# ابحث عن: "embedding model migration reindex cost"
# https://www.sbert.net/docs/pretrained_models.html
# ────────────────────────────────────────────────────────────────────

# TODO: Query the index with vectors from a different model of the same dimension and count how many queries still land on the correct document.
# مهمة: استعلم الفهرس بمتّجهات من نموذج مختلف بالبُعد نفسه، وعُدّ كم استعلامًا ما زال يصيب الوثيقة الصحيحة.
for i, (query, gold, _) in enumerate(PROBE_QUERIES):
    print(f"{query:<38} gold {gold}  right model {CHUNKS.doc_id[probe_ids[i][0]]}  "
          f"wrong model {CHUNKS.doc_id[wrong_ids[i][0]]}")
print(f"\ncorrect top-1: {RIGHT_CORRECT}/5 with the index's own model, "
      f"{MISMATCHED_CORRECT}/5 with the other one. No exception was raised.")
# TODO: Extrapolate the embedding rate to 2,000,000 documents and print the hours.
# مهمة: استقرِ معدّل التضمين إلى مليونَي وثيقة واطبع الساعات.
print(f"\n{RATE:.0f} chunks/second on this machine → re-embedding 2,000,000 documents "
      f"takes {REEMBED_HOURS:.1f} hours, once, before anyone can search again.")

## Save the artefacts

Two things go to disk. `index_bench.parquet` is the benchmark table — every index, every `nprobe`,
its recall and its query time — and it is the evidence for whatever configuration you end up
defending. `chroma_store/` is the store itself, which Wednesday and Friday both open.

The store is gitignored, and a reference copy lives in `shared/solutions_cache/`. Wednesday falls
back to that copy if you did not finish today, so nobody is blocked.

<div dir="rtl" align="right">

## احفظ المُخرَجات

يذهب شيئان إلى القرص. `index_bench.parquet` جدول القياس — كل فهرس وكل `nprobe` واستدعاؤه وزمنه —
وهو الدليل على أي ضبط تدافع عنه في النهاية. و`chroma_store/` هو المخزن نفسه، يفتحه الأربعاء والجمعة
كلاهما.

والمخزن مستثنى من git، ونسخة مرجعية منه في `shared/solutions_cache/`. ويرجع إليها الأربعاء إن لم
تُكمل اليوم، فلا يتوقّف أحد.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

BENCH = pd.concat([
    pd.DataFrame([{"corpus": "policy_docs", "index": "IndexFlatL2", "nlist": None,
                   "nprobe": None, "recall_at_5": 1.0, "query_ms": FLAT_MS,
                   "build_seconds": 0.0, "vectors": int(FLAT.ntotal)}]),
    REAL_BENCH,
    pd.DataFrame([{"corpus": "synthetic_100k", "index": "IndexFlatL2", "nlist": None,
                   "nprobe": None, "recall_at_5": 1.0, "query_ms": BIG_FLAT_MS,
                   "build_seconds": 0.0, "vectors": 100_000}]),
    BIG_BENCH,
], ignore_index=True)
BENCH["strategy"] = WINNER
BENCH.to_parquet(ARTEFACT_DIR / "index_bench.parquet", index=False)
CHUNKS.to_parquet(ARTEFACT_DIR / "indexed_chunks.parquet", index=False)

print(f"index_bench.parquet — {len(BENCH)} rows")
print(f"chroma_store/       — {REOPENED_COUNT} chunks at {STORE_PATH}")
print(BENCH[["corpus", "index", "nprobe", "recall_at_5", "query_ms"]].round(3).to_string(index=False))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(WARM_Q2_WRONG and clustered(Q2, nprobe=2)[0] == brute_force(Q2)[0],
      f"the warm-up must reproduce the slide: at nprobe=1 the clustered search returns "
      f"{clustered(Q2, 1)[0]} while brute force returns {brute_force(Q2)[0]}, and nprobe=2 fixes "
      f"it. Without that failure the rest of the lab is a demonstration of nothing",
      f"يجب أن يعيد الإحماء ما في الشريحة: عند `nprobe=1` يعيد البحث المعنقَد "
      f"{clustered(Q2, 1)[0]} بينما يعيد الشامل {brute_force(Q2)[0]}، ويصلحه `nprobe=2`. وبلا هذا "
      f"الإخفاق يصير باقي المعمل عرضًا لا شيء فيه")

check(TRUNCATED == 0,
      f"{TRUNCATED} of {len(CHUNKS)} chunks exceed the model's {model.max_seq_length}-token limit "
      f"and lose their tails silently. On this corpus the longest chunk is "
      f"{TOKEN_LENGTHS.max()} tokens, so the answer should be zero — if it is not, your chunk "
      f"size is above what the embedder can read",
      f"{TRUNCATED} من {len(CHUNKS)} مقطعًا تتجاوز حدّ {model.max_seq_length} رمزًا للنموذج وتفقد "
      f"أذنابها بصمت. وأطول مقطع في هذه المُدوّنة {TOKEN_LENGTHS.max()} رمزًا، فالجواب المفترض صفر — "
      f"وإلا فحجم مقطعك أكبر مما يقرؤه المُضمِّن")

check(bool(REAL_BENCH.recall_at_5.is_monotonic_increasing)
      and float(REAL_BENCH.recall_at_5.iloc[-1]) == 1.0,
      f"recall@5 must rise with nprobe and reach 1.0 when every list is probed — got "
      f"{REAL_BENCH.recall_at_5.round(3).tolist()} at nprobe {REAL_BENCH.nprobe.tolist()}. "
      f"Probing every list is brute force, so anything below 1.0 there is a bug, not a trade-off",
      f"يجب أن يرتفع الاستدعاء عند ٥ مع `nprobe` وأن يبلغ ١٫٠ حين تُفحص كل القوائم — والناتج "
      f"{REAL_BENCH.recall_at_5.round(3).tolist()} عند {REAL_BENCH.nprobe.tolist()}. وفحص كل "
      f"القوائم هو البحث الشامل، فما دون ١٫٠ هناك خلل لا مقايضة")

check(AGREEMENT == len(CHECK_QUESTIONS) and REOPENED_COUNT == len(CHUNKS) and REOPENED_MATCH,
      f"Chroma must agree with brute-force FAISS on all {len(CHECK_QUESTIONS)} test queries "
      f"(got {AGREEMENT}) and must return the same results after a fresh client opens the same "
      f"path (reopened {REOPENED_COUNT} of {len(CHUNKS)} chunks, same results: {REOPENED_MATCH})",
      f"يجب أن يتّفق Chroma مع FAISS الشامل في الاستعلامات {len(CHECK_QUESTIONS)} كلها "
      f"(والناتج {AGREEMENT})، وأن يعيد النتائج نفسها بعد فتح المسار نفسه بعميل جديد "
      f"(أُعيد فتح {REOPENED_COUNT} من {len(CHUNKS)} مقطعًا، والنتائج نفسها: {REOPENED_MATCH})")

FILTER_HURT = not any(cid.startswith(gold_doc) for cid in FILTER_RUNS["where doc_id=POL-108"])
check(all(cid.startswith(gold_doc) for cid in FILTER_RUNS[f"where doc_id={gold_doc}"])
      and FILTER_HURT,
      f"the metadata filter must be respected in both directions: every result under "
      f"doc_id={gold_doc} comes from that document, and filtering to POL-108 removes the answer "
      f"entirely without raising anything (answer removed: {FILTER_HURT})",
      f"يجب أن يُحترم مرشّح البيانات الوصفية في الاتجاهين: كل نتيجة تحت `doc_id={gold_doc}` من تلك "
      f"الوثيقة، والترشيح إلى `POL-108` يحذف الإجابة كليًّا بلا أي خطأ (حُذفت: {FILTER_HURT})")

POL114 = RETRIEVER_TABLE[RETRIEVER_TABLE["query"] == "policy POL-114"].iloc[0]
RETURNS = RETRIEVER_TABLE[RETRIEVER_TABLE["query"] == "how long do I have to return an item"].iloc[0]
check(bool(POL114.bm25_correct and not POL114.dense_correct
           and RETURNS.dense_correct and not RETURNS.bm25_correct),
      f"the two retrievers must fail in opposite directions — BM25 ranks POL-114 first for "
      f"'policy POL-114' (dense gave {POL114.dense_top1}) and dense ranks POL-101 first for the "
      f"return-window question (BM25 gave {RETURNS.bm25_top1}). That opposition is the argument "
      f"for hybrid retrieval on Thursday",
      f"يجب أن يُخفق المُسترجِعان في اتجاهين متضادّين — يرتّب BM25 وثيقة POL-114 أولًا لاستعلام "
      f"'policy POL-114' (وأعطى الكثيف {POL114.dense_top1})، ويرتّب الكثيف POL-101 أولًا لسؤال "
      f"نافذة الإرجاع (وأعطى BM25 {RETURNS.bm25_top1}). وهذا التضادّ هو حجّة الاسترجاع الهجين يوم "
      f"الخميس")

check(MISMATCHED_CORRECT < RIGHT_CORRECT,
      f"querying the index with another model's vectors must degrade the results — got "
      f"{MISMATCHED_CORRECT}/5 correct against {RIGHT_CORRECT}/5 with the right model. If they "
      f"match, the two models are more alike than the demonstration needs",
      f"يجب أن تتدهور النتائج عند استعلام الفهرس بمتّجهات نموذج آخر — والناتج {MISMATCHED_CORRECT}/5 "
      f"مقابل {RIGHT_CORRECT}/5 بالنموذج الصحيح. وإن تساويا فالنموذجان أشبه ببعضهما مما يحتاجه العرض")

report()

## What's next

**Tomorrow the model arrives.** `chroma_store/` gets a language model in front of it, and the pair
is retrieval-augmented generation. You will watch a model answer one of these questions with no
context — confidently, and wrongly — and then watch the same question answered correctly from four
chunks you retrieved with the store you built today.

The prompt goes through four versions and the interesting one is the third: a single added
sentence turns a system that invents answers to out-of-scope questions into one that declines them.

**A6 is due today.**

<div dir="rtl" align="right">

## ما التالي

**غدًا يصل النموذج.** يقف نموذج لغوي أمام `chroma_store/`، ويكون الاثنان توليدًا معزّزًا بالاسترجاع.
وسترى نموذجًا يجيب عن أحد هذه الأسئلة بلا سياق — بثقة وبخطأ — ثم ترى السؤال نفسه يُجاب صحيحًا من
أربعة مقاطع استرجعتها بالمخزن الذي بنيته اليوم.

ويمرّ الموجّه بأربع نسخ، وأهمّها الثالثة: جملة واحدة تُضاف فتحوّل نظامًا يخترع إجابات للأسئلة خارج
النطاق إلى نظام يرفضها.

**ويُسلَّم الواجب السادس اليوم.**

</div>